In [1]:
import json
import random
import base64
import os
import shutil
import subprocess
import dill
import nltk
import config
import itertools
import io
import math
import textwrap
import taxonomy
import time

from PIL import Image
from utils.DataHandlers import LatexDataHandler
from utils.ModelClient import ModelClient
from pathlib import Path
from collections import Counter
from pdf2image import convert_from_bytes
from pydantic import BaseModel

In [2]:
EDIT_TYPE = config.EDIT_TYPE
DATA_FOLDER_PATH = config.DATA_FOLDER_PATH
TASK_ID = 'judge'
COMPILATION_TYPE = config.COMPILATION_TYPE

In [3]:
GROUND_TRUTH_TYPE = 'generate_ground_truth_description'

In [4]:
def get_system_prompt():
    system_prompt = textwrap.dedent("""
    Compare the following: a generated review of weaknesses in a scientific paper and a ground truth weakness. Assess how well the generated review captures the ground truth weakness in terms of relatedness and specificity. Do not penalize the generated review for mentioning additional weaknesses or details not contained in the ground truth.
                                
    Rating Criteria:
    - 5 points: The generated review covers all key points from the ground truth weakness with a similar level of detail.
    - 4 points: The generated review covers all key points from the ground truth weakness, but it is less detailed than the ground truth.
    - 3 points: The generated review is related to the ground truth, but does not cover all key points.
    - 2 points: The generated review is only loosely related to the ground truth.
    - 1 point: There is no connection between the generated review and the ground truth.
                                    
    Assign a rating (1-5).
    """[1:]
    )

    return system_prompt

In [5]:
SYSTEM_PROMPT = get_system_prompt()

MESSAGE_TEMPLATE = """
Ground-truth weakness: 
{0}

Generated review of weaknesses:
{1}
"""

In [6]:
class Score(BaseModel):
    final_score: int

In [7]:
def manuscript_iterator(project_data):
    for paper_id, error_id, data in project_data:
        meta_id = f'{paper_id}___{error_id}' #combined paper + error id
        ground_truth = None
        for gr in data['ground_truth']:
            if gr['task_id'] == GROUND_TRUTH_TYPE:
                ground_truth = gr['response_text']
                break

        for model_res in data['model_responses']:
            model_id = model_res['model_id'] 
            task_id = model_res['task_id']
            input_type = model_res['input_type']
            response = model_res['response_text']

            meta_id = f'{paper_id}___{error_id}___{model_id}___{task_id}___{input_type}'

            message = MESSAGE_TEMPLATE.format(
                ground_truth,
                response,
            )

            req_data = {
                'system_prompt': SYSTEM_PROMPT,
                'message': message,
                'reasoning_effort': "high", 
                'structured_output_format': Score,
            }

            yield (meta_id, req_data)

In [8]:
model_client = ModelClient("openai", "gpt-5.4-2026-03-05", "gpt_54", "output/judge_gpt54_workspace")

In [9]:
with open('runtime.txt', 'w') as fp:
    ts = time.time()
    fp.write(str(ts))

In [10]:
input_path = Path('./input.dill')
if not input_path.is_file():
    model_responses = {}
    for filename in os.listdir('responses'):
        with open(f'./responses/{filename}', 'rb') as fp:
            model_res = dill.load(fp)
            for paper_id, error_id, res in model_res:
                meta_id = f'{paper_id}___{error_id}'
                if meta_id in model_responses:
                    if 'model_responses' in model_responses[meta_id] and 'model_responses' in res:
                        model_responses[meta_id]['model_responses'].extend(res['model_responses'])
                else:
                    model_responses[meta_id] = res

    with open(f'./ground_truth/responses_generate_ground_truth_description_latex_gpt_54.dill', 'rb') as fp:
        ground_truth = dill.load(fp)
        for paper_id, error_id, res in ground_truth:
            meta_id = f'{paper_id}___{error_id}'
            if 'ground_truth' in res:
                model_responses[meta_id]['ground_truth'] = res['ground_truth']

    final_responses = []
    for meta_id, res in model_responses.items():
        paper_id, error_id = meta_id.split('___')
        final_responses.append((paper_id, error_id, res))

    with open(input_path, 'wb') as fp:
        dill.dump(final_responses, fp)

In [ ]:
project_data = []
with open('./input.dill', 'rb') as fp:
    compiled_data = dill.load(fp)

    for project_id, error_id, compiled in compiled_data:
        if compiled['valid'] and compiled["edited_pdf_path"] is not None and 'model_responses' in compiled:
            project_data.append((project_id, error_id, compiled))

# manuscript_file_paths = [project_data[2] for project_data in project_ids]
cache_path = f'{config.OUTPUT_FOLDER_PATH}responses_{TASK_ID}_{model_client.model_prefix_name}_cache'

for meta_id, res in model_client.iterative_query(manuscript_iterator(project_data), cache_file_path=cache_path, batch=True):
    paper_id, error_id, model_id, task_id, input_type  = meta_id.split('___')
    print(meta_id)

    for compiled in compiled_data:
        if compiled[0] == paper_id and compiled[1] == error_id and 'model_responses' in compiled[2]:
            model_responses = compiled[2]['model_responses']
            for model_res in model_responses:
                if model_res['model_id'] == model_id and model_res['task_id'] == task_id and model_res['input_type'] == input_type:
                    model_res['judgement'] = res

with open(f'{config.OUTPUT_FOLDER_PATH}responses_{TASK_ID}_{model_client.model_prefix_name}.dill', 'wb') as fp:
    dill.dump(compiled_data, fp)

Loading from cache...
Loading cached responses from output/responses_judge_gpt_54_cache.dill
2412.17534___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0___gemini_25_flash_lite___identification_and_localization___latex
2412.17534___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0___gpt_54___identification_and_localization___images
2412.17534___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0___gpt_54___identification_and_localization___images_and_latex
2412.17534___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0___gpt_54___identification_and_localization___latex
2412.17534___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0___gpt_54_mini___identification_and_localization___images
2412.17534___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0___gpt_54_mini___identification_and_localization___images_and_latex
2412.17534___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0___gpt_54_mini___identification_and_localization___latex
2412.17534___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0___gpt_54_nano___identification_and_localization___images
2412.17534___bWV0aG9kb2xvZ3lfcmVzdWx0X2N

/Users/aadi.joshi/miniconda3/envs/general/lib/python3.13/site-packages/dill/_dill.py:422: PicklingWarning: Cannot locate reference to <class '__main__.Score'>.
  StockPickler.save(self, obj, save_persistent_id)
/Users/aadi.joshi/miniconda3/envs/general/lib/python3.13/site-packages/dill/_dill.py:422: PicklingWarning: Cannot pickle <class '__main__.Score'>: __main__.Score has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)
/Users/aadi.joshi/miniconda3/envs/general/lib/python3.13/site-packages/dill/_dill.py:422: PicklingWarning: Cannot locate reference to <class '__main__.Score'>.
  StockPickler.save(self, obj, save_persistent_id)
/Users/aadi.joshi/miniconda3/envs/general/lib/python3.13/site-packages/dill/_dill.py:422: PicklingWarning: Cannot pickle <class '__main__.Score'>: __main__.Score has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)
/Users/aadi.joshi/miniconda3/e

unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
unable to parse
2505.07968___c3Vib3B0aW1hbF9tZXRob2RvbG9neQ==___gemini_25_flash___identification_and_localization___images_and_latex
2505.07968___c3Vib3B0aW1hbF9tZXRob2RvbG9neQ==___gemini_25_flash___identification_and_localization___latex
2505.07968___c3Vib3B0aW1hbF9tZXRob2RvbG9neQ==___gemini_25_flash_lite___identification_and_localization___images
2505.07968___c3Vib3B0aW1hbF9tZXRob2RvbG9neQ==___gemini_25_flash_lite___identification_and_localization___images_and_latex
2505.07968___c3Vib3B0aW1hbF9tZXRob2RvbG9neQ==___gemini_25_flash_lite___identifi

In [12]:
# make sure all model responses are recorded
with open(f'{config.OUTPUT_FOLDER_PATH}responses_{TASK_ID}_{model_client.model_prefix_name}.dill', 'rb') as fp:
    responses = dill.load(fp)

count = 0
incomplete = 0
for mid, errid, res in responses:
    if 'model_responses' in res:
        for model_response in res['model_responses']:
            if 'judgement' not in model_response:
                incomplete += 1
                continue
            count += 1

print(f'Total responses: {count}')
print(f'Number of missing model responses: {incomplete}') 

Total responses: 3871
Number of missing model responses: 142
